# 01｜目标检测是什么：从图像分类走向物体定位

我们暂时离开 DETR 的具体结构，先回到所有目标检测模型共同面对的问题。

这一课只回答一个问题：

> 给模型一张图片，我们究竟希望它输出什么？

这个问题没有想清楚，后面的 YOLO、Anchor、NMS、Object Query 和匈牙利匹配都会变成孤立名词。

## 1. 为什么要退回来学习任务本身

以前做 MNIST 或 CIFAR-10 分类时，一张图片最终对应一个类别。模型只需要回答“整张图片最像什么”。

目标检测不同。一张真实图片中可能同时出现人、汽车、狗和交通灯，并且每种物体还可能出现多次。模型不仅要认出它们，还要指出每个物体的位置。

因此，目标检测并不是简单地给分类模型多加几个类别。它同时引入了三个新问题：

1. 图片里有什么？
2. 每个物体在哪里？
3. 图片里一共有多少个物体？

后面所有检测模型，都是在用不同方法回答这三个问题。

## 2. 从一个具体场景开始

假设有一张街道照片，里面包含：

- 左侧一名行人。
- 中间一辆汽车。
- 右侧另一名行人。

面对同一张图片，不同视觉任务会提出不同要求：

| 任务 | 问题 | 可能的输出 |
|---|---|---|
| 图像分类 | 这张图主要是什么场景或类别？ | 街道 |
| 多标签分类 | 这张图出现过哪些类别？ | 人、汽车 |
| 单目标定位 | 主要物体是什么，在哪里？ | 汽车 + 一个框 |
| 目标检测 | 每一个物体是什么，分别在哪里？ | 两个人框 + 一个汽车框 |

注意：多标签分类虽然知道图片里有“人”和“汽车”，但不知道有两个人，也不知道它们分别位于哪里。

## 3. 图像分类：一张图片对应一个类别分布

假设一共有 $K$ 个类别，分类模型通常输出：

$$
\text{logits}\in\mathbb{R}^{B\times K}
$$

其中每张图片对应 $K$ 个类别分数。经过 Softmax 后，可以得到一个类别概率分布。

对于一张图片，分类模型只需要选出一个主要类别：

$$
\text{整张图片}\rightarrow\text{一组类别分数}\rightarrow\text{一个类别}
$$

分类模型不会告诉我们物体占据了哪些像素，也无法分别表示同类的多个物体。

## 4. 单目标定位：类别之外，再预测一个边界框

如果规定每张图片只有一个主要物体，模型可以同时输出：

- 物体类别。
- 一个描述物体位置的矩形框。

这类任务可以称为单目标定位。它的输出可以理解为：

$$
(\text{class},\text{box})
$$

例如：

$$
\begin{aligned}
\text{类别}&:\text{猫}\\
\text{边界框}&:(80,50)\rightarrow(300,260)
\end{aligned}
$$

它已经比分类多了“在哪里”，但仍然假设只有一个主要物体。

## 5. 目标检测：输出数量不固定的多个结果

目标检测取消了“只有一个物体”的假设。图片里有几个物体，就应该输出几条检测结果。

每条检测结果都包含：

$$
(\text{类别},\text{边界框})
$$

一张图片的完整标注可以写成：

$$
Y=\{(c_1,b_1),(c_2,b_2),\ldots,(c_M,b_M)\}
$$

其中：

- $M$：这张图片里的真实物体数量。
- $c_i$：第 $i$ 个物体的类别。
- $b_i$：第 $i$ 个物体的边界框。

不同图片的 $M$ 可以不同，这就是检测任务比分类任务多出的核心困难之一。

## 6. “目标识别”和“目标检测”有什么区别

日常交流中，人们有时会把它们混着说。但在学习模型时，最好区分：

- **图像识别或物体识别**：更宽泛，重点是判断看到了什么。
- **目标检测**：不仅识别类别，还要用边界框定位每个目标。

YOLO 和 DETR 都属于目标检测模型。它们输出的不只是“有一辆车”，而是“这辆车是什么，并且它位于这个矩形区域”。

后面的笔记统一使用“目标检测”这个更准确的名称。

## 7. 边界框是什么

边界框（Bounding Box）是包围物体的矩形。它用少量数字近似描述物体在图片中的位置和大小。

为什么使用矩形框，而不是直接描出物体轮廓？

- 矩形框标注成本较低。
- 只需要 4 个坐标就能表示。
- 足以回答许多“物体在哪里”的问题。
- 方便模型预测和比较。

边界框并不等于物体的真实轮廓。框内可能包含一些背景，物体的某些斜角区域也不会被精确贴合。精确到像素轮廓的任务属于图像分割，后面不要把两者混淆。

## 8. 先建立图像坐标系

在常见图像坐标系中：

- 左上角是原点 $(0,0)$。
- $x$ 轴向右增大。
- $y$ 轴向下增大。
- 图片宽度记为 $W$。
- 图片高度记为 $H$。

$$
\begin{aligned}
(0,0)&=\text{左上角}\\
+x&:\text{向右}\\
+y&:\text{向下}
\end{aligned}
$$

这与数学课中常见的 $y$ 轴向上不同。以后看到边界框坐标时，要先确认使用的是图像坐标系。

## 9. 边界框格式一：左上角与右下角

一种常见表示方法是：

$$
(x_{min},y_{min},x_{max},y_{max})
$$

它也常被简称为 `xyxy`：

- $(x_{min},y_{min})$：矩形左上角。
- $(x_{max},y_{max})$：矩形右下角。

框的宽和高为：

$$
w=x_{max}-x_{min}
$$

$$
h=y_{max}-y_{min}
$$

这种格式非常适合计算两个框的交集区域，后面学习 IoU 时会再次使用。

## 10. 边界框格式二：中心点、宽和高

另一种常见表示方法是：

$$
(c_x,c_y,w,h)
$$

它也常被简称为 `cxcywh`：

- $c_x$：边界框中心点的横坐标。
- $c_y$：边界框中心点的纵坐标。
- $w$：边界框宽度。
- $h$：边界框高度。

原始 DETR 的边界框预测常使用归一化后的 `cxcywh`。YOLO 的许多表示中也会出现中心点、宽和高，但具体参数化与解码方式要根据版本确认。

两种格式表达的是同一个矩形，并不是两类不同的边界框。

## 11. 两种边界框格式怎样转换

从 `xyxy` 转成 `cxcywh`：

$$
c_x=\frac{x_{min}+x_{max}}{2},\qquad c_y=\frac{y_{min}+y_{max}}{2}
$$

$$
w=x_{max}-x_{min},\qquad h=y_{max}-y_{min}
$$

从 `cxcywh` 转回 `xyxy`：

$$
x_{min}=c_x-\frac{w}{2},\qquad x_{max}=c_x+\frac{w}{2}
$$

$$
y_{min}=c_y-\frac{h}{2},\qquad y_{max}=c_y+\frac{h}{2}
$$

学习检测代码时，经常需要在两种格式之间转换，所以关键不是死记名称，而是能画出矩形并解释四个数。

## 12. 用一个具体边界框算一遍

假设图片大小是 $640\times480$，也就是宽 $W=640$、高 $H=480$。某个物体的 `xyxy` 边界框是：

$$
(x_{min},y_{min},x_{max},y_{max})=(160,120,480,360)
$$

先计算宽和高：

$$
w=480-160=320,\qquad h=360-120=240
$$

再计算中心点：

$$
c_x=\frac{160+480}{2}=320,\qquad c_y=\frac{120+360}{2}=240
$$

所以同一个框的 `cxcywh` 表示为：

$$
(320,240,320,240)
$$

不要被重复的数字迷惑，这只是当前例子恰好让框位于图片中央，并占图片宽高的一半。

## 13. 为什么经常把边界框归一化

像素坐标会随图片大小变化。同一个相对位置，在 $640\times480$ 图片和 $1280\times960$ 图片中会对应不同像素值。

归一化就是分别除以图片宽度和高度，使坐标通常落在 0 到 1 之间。对于 `cxcywh`：

$$
\hat{c}_x=\frac{c_x}{W},\qquad \hat{c}_y=\frac{c_y}{H},\qquad \hat{w}=\frac{w}{W},\qquad \hat{h}=\frac{h}{H}
$$

上一节例子的归一化结果是：

$$
(0.5,0.5,0.5,0.5)
$$

它表示：框中心位于图片宽和高的 50% 处，框本身占图片宽度的 50%、高度的 50%。

归一化让不同尺寸图片上的位置和大小进入相近数值范围，也让表示更关注相对位置。

## 14. 多个目标为什么构成一个无序集合

假设一张图里有一只猫和一只狗。标注可以先写猫再写狗，也可以先写狗再写猫：

$$
\{(\text{猫},b_1),(\text{狗},b_2)\}
$$

与

$$
\{(\text{狗},b_2),(\text{猫},b_1)\}
$$

表示完全相同的检测结果。

这和句子不同。句子中的词序改变可能改变含义；检测标注的排列顺序通常没有意义。

现在只需要建立这个事实。以后学习 DETR 时，我们会看到：正因为真实目标是无序集合，模型训练时才需要解决“哪个预测与哪个真实目标对应”的问题。

## 15. 从分类到检测，困难究竟增加在哪里

把变化放在一起看：

| 问题 | 图像分类 | 目标检测 |
|---|---|---|
| 输出数量 | 每张图通常一个结果 | 每张图数量不固定 |
| 输出内容 | 类别 | 每个目标的类别和边界框 |
| 空间位置 | 通常不直接输出 | 必须预测 |
| 同类多个实例 | 不需要区分 | 必须分别定位 |
| 结果顺序 | 一个类别分布 | 多个目标构成无序集合 |

因此，目标检测至少要同时解决：

1. **分类问题**：这个框里的物体是什么？
2. **回归问题**：这个框的四个连续坐标应该是多少？
3. **数量问题**：应该输出多少个有效目标？
4. **实例问题**：怎样把同类别的不同物体分开？

后续课程中的每一个机制，都能回到这四类问题中找到自己的位置。

## 16. 与已经学过的知识怎样连接

### 与 CNN 分类的连接

CNN 分类模型已经会提取图像特征，并根据特征判断类别。检测模型仍然需要这种能力，但不能过早把整张特征图压缩成一个向量，因为空间位置还要用于预测边界框。

### 与损失函数的连接

以前的分类损失负责判断类别对不对。检测还要增加边界框回归损失，用来衡量预测位置与真实位置之间的差距。

### 与 Transformer 的连接

Transformer 能让不同图像区域交换信息，但它不会自动解决边界框表示、预测数量和目标对应关系。DETR 仍然必须专门设计这些检测环节。

## 17. 本课暂时不回答的问题

学到这里，自然会出现一些新问题：

- 模型预测的框与真实框有多接近，怎样量化？
- 为什么同一个物体会被预测出许多重复框？
- 模型怎样知道哪个位置应该负责哪个物体？
- 一张图片的物体数量不固定，神经网络怎样输出规则张量？
- YOLO 和 DETR 分别怎样解决这些问题？

这些都很重要，但现在不同时展开。

下一课只解决第一个问题：**如何用 IoU 衡量两个边界框的重叠程度。** 先学会评价一个框，再讨论模型怎样生成框。

## 18. 本节小结

这一课需要真正记住六个结论：

1. 图像分类回答整张图是什么，目标检测回答每个物体是什么、在哪里。
2. 一条检测结果至少包含一个类别和一个边界框。
3. `xyxy` 用左上角和右下角表示框，`cxcywh` 用中心点、宽和高表示框。
4. 图像坐标原点通常在左上角，$x$ 向右、$y$ 向下。
5. 边界框可以除以图片宽高进行归一化，使坐标通常位于 0 到 1。
6. 一张图片中的多个目标构成数量不固定、排列顺序无关的集合。

最核心的任务变化是：

$$
\begin{aligned}
\text{图像分类：图片}&\rightarrow\text{一个类别}\\
\text{目标检测：图片}&\rightarrow\{(\text{类别}_1,\text{框}_1),(\text{类别}_2,\text{框}_2),\ldots\}
\end{aligned}
$$

## 19. 自测问题

1. 图像分类、多标签分类和目标检测分别能回答什么问题？
2. 为什么多标签分类不能替代目标检测？
3. 一条完整的目标检测结果至少包含哪两部分？
4. 为什么不同图片的检测结果数量可以不同？
5. 常见图像坐标系的原点在哪里？$x$ 和 $y$ 分别向哪里增大？
6. `xyxy` 中四个值分别是什么？
7. `cxcywh` 中四个值分别是什么？
8. 已知 `xyxy=(10,20,50,80)`，框的宽和高是多少？
9. 上一题中，框中心点坐标是多少？
10. 为什么同一个框可以同时写成 `xyxy` 和 `cxcywh`？
11. 对坐标进行归一化有什么作用？
12. 为什么检测结果通常被看成无序集合？
13. 从分类进入检测后，增加了哪几类核心困难？

### 自测参考答案

1. 分类判断整张图的主要类别；多标签分类判断出现过哪些类别；检测还要分别定位每个物体。
2. 它不知道每类物体出现了几次，也不给出每个实例的位置。
3. 物体类别和边界框。
4. 不同图片包含的真实物体数量不同。
5. 原点通常在左上角，$x$ 向右增大，$y$ 向下增大。
6. 左上角 $(x_{min},y_{min})$ 和右下角 $(x_{max},y_{max})$。
7. 中心点 $(c_x,c_y)$、宽 $w$ 和高 $h$。
8. $w=50-10=40$，$h=80-20=60$。
9. $c_x=(10+50)/2=30$，$c_y=(20+80)/2=50$。
10. 它们只是同一个矩形的两种坐标表示，可以通过公式互相转换。
11. 让不同图片尺寸上的框使用统一的相对位置和相对大小范围。
12. 调换目标的书写顺序不会改变图片中有哪些物体。
13. 类别判断、边界框回归、输出数量变化，以及同类不同实例的区分。